# 01 — Training & Exploring Vision Transformer (ViT-Tiny) on STL-10

Accelerated training and validation pipeline on Google Colab (T4 GPU) for our from-scratch PyTorch implementation of the Vision Transformer (**vit-under-the-hood**).

### Architecture & Equation Mapping (Dosovitskiy et al., 2020)

| Paper Formulation | Mathematical Role | Repository Implementation |
| :--- | :--- | :--- |
| **Eq. 1**: $\mathbf{z}_0 = [\mathbf{x}_{\text{class}}; \mathbf{x}_p^1\mathbf{E}; \dots; \mathbf{x}_p^N\mathbf{E}] + \mathbf{E}_{pos}$ | Patch projection, `[CLS]` token, 1D position embeddings | `ViTEmbedding` (`src/models/embeddings.py`) |
| **Eq. 2**: $\mathbf{z}'_l = \text{MSA}(\text{LN}(\mathbf{z}_{l-1})) + \mathbf{z}_{l-1}$ | Scaled dot-product multi-head attention (Pre-LN) | `MultiHeadAttention` (`src/models/attention.py`) |
| **Eq. 3**: $\mathbf{z}_l = \text{MLP}(\text{LN}(\mathbf{z}'_l)) + \mathbf{z}'_l$ | Feed-Forward network with GELU non-linearity | `MLP` & `TransformerEncoderBlock` (`src/models/transformer.py`) |
| **Eq. 4**: $\mathbf{y} = \text{LN}(\mathbf{z}_L^0)$ | Final layer normalization and classification head | `VisionTransformer.head` (`src/models/vit.py`) |

> **Inductive Bias & Learning Dynamics Note:**
> Convolutional networks inherently encode translation equivariance and local spatial priors (pixels close together are related). Vision Transformers possess no such architectural bias; self-attention is permutation-equivariant and initially agnostic to 2D image topology. On compact training sets like STL-10 (5,000 labeled samples), from-scratch convergence critically depends on:
> 1. **Parameter sizing** (`vit_tiny`, ~3.63M params to avoid massive overfitting).
> 2. **Heavy spatial augmentations** (RandomCrop, Flips, ColorJitter) to force invariant feature learning.
> 3. **Linear warmup** (preventing early gradient explosion & rank collapse).
> 4. **Label smoothing** regularizing overconfident class logits.

## 1. Hardware & Environment Setup

Verify GPU accelerator availability (Tesla T4 recommended) and PyTorch environment.

In [ ]:
!nvidia-smi

import matplotlib.pyplot as plt
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Active Device:   {device}")
if device == "cuda":
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")

## 2. Clone Repository & Install Dependencies

Clone the `vit-under-the-hood` repository and install dependencies in editable mode.

In [ ]:
import os

# Clone repository or pull latest updates if already present
if not os.path.exists("vit-under-the-hood") and not os.path.exists("src"):
    !git clone https://github.com/diallo-aliou/vit-under-the-hood.git
    %cd vit-under-the-hood
elif os.path.exists("vit-under-the-hood"):
    %cd vit-under-the-hood
    !git pull
else:
    !git pull

# Install dependencies in editable mode
!pip install -r requirements.txt
!pip install -e .

### 2.1 Mount Google Drive for Checkpoint Persistence (Recommended)

Mounting your Google Drive ensures that checkpoints (`outputs/checkpoints/best_model.pth` and `epoch_*.pth`) and training curves are permanently saved. If your Colab runtime disconnects, the notebook can restore them instantly without re-training!

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/vit-under-the-hood/outputs'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Google Drive mounted successfully! Checkpoints will persist in: {DRIVE_DIR}')
except Exception as e:
    DRIVE_DIR = None
    print(f'Running without Google Drive mount: {e}')

## 3. Visualizing the STL-10 Dataset & Data Augmentations

Before feeding pixels into self-attention matrices, let's look directly at our data.

STL-10 consists of 10 distinct classes: `airplane`, `bird`, `car`, `cat`, `deer`, `dog`, `horse`, `monkey`, `ship`, `truck`. Each image is $96 \times 96$ pixels (3 channels).

Below, we visualize:
1. **Real Dataset Samples**: A grid of images from the training set with natural un-normalized RGB colors.
2. **Data Augmentation Dynamics**: Stochastic transformations applied during training to regularize the Vision Transformer.

In [ ]:
import torchvision.transforms as T

from src.training.dataset import STL10_CLASSES, get_stl10_dataloaders
from src.utils.visualize import plot_augmentation_examples, plot_dataset_samples

# Load STL-10 DataLoaders
train_loader, test_loader = get_stl10_dataloaders(
    data_dir="./data",
    batch_size=16,
    num_workers=2,
    download=True,
)

# 1. Visualize real dataset samples with their class labels
batch_images, batch_labels = next(iter(train_loader))
plot_dataset_samples(batch_images, batch_labels, STL10_CLASSES, num_samples=8, nrows=2)

# 2. Visualize stochastic data augmentations on a real image
aug_pipeline = T.Compose([
    T.RandomResizedCrop(96, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2),
])
plot_augmentation_examples(batch_images[0], aug_pipeline, num_examples=4)

## 4. Patch Extraction & Linear Projection on a Real Image

A standard Transformer expects a sequence of 1D vectors $(\mathbf{z} \in \mathbb{R}^{N \times D})$. To feed a 2D image $(\mathbf{x} \in \mathbb{R}^{3 \times H \times W})$ into a Transformer, we divide the image into non-overlapping patches of size $P \times P$:

$$N = \frac{H \cdot W}{P^2} = \frac{96 \cdot 96}{8^2} = 12 \times 12 = 144 \text{ patches}$$

Each $8 \times 8 \times 3$ patch has $192$ values. A learnable linear projection maps each flattened patch to the embedding dimension $D=192$, prepends a learnable `[CLS]` token ($N+1 = 145$), and adds learnable 1D position embeddings $\mathbf{E}_{pos} \in \mathbb{R}^{145 \times 192}$.

In [ ]:
from src.models.embeddings import ViTEmbedding
from src.utils.visualize import plot_patch_grid

# Extract a sample from test set
test_batch_images, test_batch_labels = next(iter(test_loader))
sample_image = test_batch_images[0:1]

# 1. Visualize spatial patch decomposition on a real STL-10 image
plot_patch_grid(sample_image, patch_size=8, save_path="outputs/patch_grid.png")

# 2. Verify sequence output through ViTEmbedding
embedding_layer = ViTEmbedding(image_size=96, patch_size=8, embed_dim=192)
tokens = embedding_layer(sample_image)

print(f"Input image tensor:       {sample_image.shape} (B, C, H, W)")
print("Spatial patches count:    12 × 12 = 144 patches")
print(f"Sequence after [CLS]+Pos: {tokens.shape} (B, N+1, D)")

## 5. Model Architecture & Layer-by-Layer Summary

We instantiate `vit_tiny`, our compact Vision Transformer specifically sized for STL-10:
- **Image Size**: $96 \times 96$, **Patch Size**: $8 \times 8$ ($N = 144$ tokens)
- **Embedding Dim ($D$)**: $192$
- **Depth**: $8$ Transformer Encoder blocks
- **Attention Heads ($H$)**: $3$ (head dimension $d_k = 192 / 3 = 64$)
- **MLP Hidden Dim**: $192 \times 4 = 768$
- **Total Parameters**: ~3.63M (small enough to prevent catastrophic overfitting on 5k images)

In [ ]:
from torchinfo import summary

from src.models.vit import vit_tiny

model = vit_tiny(num_classes=10)
summary(
    model,
    input_size=(1, 3, 96, 96),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    depth=3,
)

## 6. Execute Training on STL-10 (50 Epochs)

We launch the training loop configured in `configs/vit_tiny_stl10.yaml`:
- **Optimizer**: AdamW ($\text{lr} = 5 \times 10^{-4}$, weight decay $0.05$)
- **Warmup**: 5 epochs linear warmup
- **Scheduler**: Cosine decay down to $\eta_{\text{min}} = 10^{-6}$
- **Label Smoothing**: $0.1$
- **Snapshots**: Every epoch checkpoint is saved to `outputs/checkpoints/epoch_*.pth` for the Phase 6 Timelapse!

> **Smart Persistence**: If a checkpoint already exists in Google Drive, the cell restores it automatically so you never have to re-train after a session disconnect.

In [ ]:
drive_ckpt = "/content/drive/MyDrive/vit-under-the-hood/outputs/checkpoints/best_model.pth"
local_ckpt = "outputs/checkpoints/best_model.pth"

if os.path.exists(drive_ckpt) and not os.path.exists(local_ckpt):
    print("Checkpoint found on Google Drive! Restoring files to skip re-training...")
    os.makedirs("outputs", exist_ok=True)
    !cp -r /content/drive/MyDrive/vit-under-the-hood/outputs/* outputs/
    print("Restoration complete! Checkpoints ready.")
elif not os.path.exists(local_ckpt):
    print("Starting GPU training pipeline (50 epochs on STL-10)...")
    !python train.py --config configs/vit_tiny_stl10.yaml --device cuda
    if os.path.exists("/content/drive/MyDrive"):
        print("Backing up checkpoints to Google Drive for permanent persistence...")
        !mkdir -p /content/drive/MyDrive/vit-under-the-hood/outputs
        !cp -r outputs/* /content/drive/MyDrive/vit-under-the-hood/outputs/
        print("Google Drive backup complete!")
else:
    print("Trained model checkpoint already available locally.")

## 7. Training Dynamics Analysis

Let's inspect the loss and accuracy curves saved during training.

In [ ]:
from PIL import Image

curves_img = Image.open("outputs/training_curves.png")
plt.figure(figsize=(14, 5))
plt.imshow(curves_img)
plt.axis("off")
plt.title("ViT-Tiny on STL-10 — Training Dynamics (Warmup + Cosine Decay)", fontsize=13, fontweight="bold")
plt.show()

## 8. Model Evaluation & Visual Predictions (Bourke Style)

Let's evaluate the trained model on unseen test images:
1. **Prediction Grid**: Visualizing test images with predicted class, confidence %, and ground truth (green if correct, red if incorrect).
2. **Top-5 Probability Distribution**: Inspecting the model's confidence distribution via a horizontal bar chart.

In [ ]:
from src.utils.visualize import plot_prediction_grid, plot_prediction_topk

# Load best checkpoint weights
checkpoint = torch.load("outputs/checkpoints/best_model.pth", map_location="cpu")
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
model.to(device)

# Grab a test batch
eval_images, eval_labels = next(iter(test_loader))

with torch.no_grad():
    batch_preds = model(eval_images.to(device)).argmax(dim=-1).cpu()
    batch_acc = (batch_preds == eval_labels).float().mean() * 100
print(f"Sample Batch Accuracy: {batch_acc:.1f}% ({int((batch_preds == eval_labels).sum())}/{len(eval_labels)} correct)")

# 1. Prediction grid with green/red status indicators
plot_prediction_grid(
    model=model,
    images=eval_images,
    labels=eval_labels,
    class_names=STL10_CLASSES,
    device=device,
    num_samples=8,
)

# 2. Top-5 confidence distribution for an individual test sample
sample_img = eval_images[0:1].to(device)
with torch.no_grad():
    sample_probs = torch.softmax(model(sample_img), dim=-1)[0]

plot_prediction_topk(
    image=sample_img,
    true_label=int(eval_labels[0].item()),
    probs=sample_probs,
    class_names=STL10_CLASSES,
    top_k=5,
)

## 9. Under the Hood: Attention Maps & Layer Progression

This is the core identity of our project: **seeing what the Vision Transformer sees**.

When an image passes through the self-attention mechanism:
$$\text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V = A \cdot V$$

The attention weight matrix $A \in \mathbb{R}^{H \times N \times N}$ defines how all tokens route information between each other.
- The first row $A[:, 0, 1:]$ represents how the `[CLS]` token attends to each of the 144 image patches.
- We reshape this $(144,)$ vector to a $(12, 12)$ 2D spatial grid and upsample it via bicubic interpolation to $(96, 96)$.

Below we visualize:
1. **Multi-Head Specialization (Final Layer)**: Head 1, Head 2, Head 3, and the mean attention overlay.
2. **Attention Progression across Layers**: How attention evolves from early layers (broad/diffuse context) to deep layers (focused semantic object localization).

In [ ]:
from src.utils.visualize import plot_attention_heads, plot_attention_layer_progression

# Forward pass with multi-layer attention extraction
test_img = eval_images[0:1].to(device)
with torch.no_grad():
    logits, all_attentions = model(test_img, return_all_attentions=True)

# 1. Multi-Head Specialization in Layer 8 (Final Layer)
plot_attention_heads(
    image=test_img,
    attention_weights=all_attentions[-1],
    layer_idx=7,
    patch_size=8,
)

# 2. Attention Depth Progression across Transformer Layers (Layer 1 -> 3 -> 5 -> 8)
plot_attention_layer_progression(
    image=test_img,
    all_attentions=all_attentions,
    layers=[0, 2, 4, 7],
    patch_size=8,
)